In [113]:
# 1. Core Data Libraries
import pandas as pd
import numpy as np

# 2. Machine Learning Pipeline Tools
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

# 3. The Algorithm & Evaluation
import xgboost as xgb
from sklearn.metrics import classification_report

In [114]:
# Load the dataset (Assuming the file is named 'WA_Fn-UseC_-Telco-Customer-Churn.csv')
data = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')

# Drop the non-predictive ID column
data = data.drop('customerID', axis=1)

# Fix the hidden blank spaces in TotalCharges and force it to numeric
data['TotalCharges'] = pd.to_numeric(data['TotalCharges'], errors='coerce')
data['TotalCharges'] = data['TotalCharges'].fillna(0)

print(f"Data cleaned. Current shape: {data.shape}")

Data cleaned. Current shape: (7043, 20)


In [115]:
# 1. Encode Target Variable manually
data['Churn'] = data['Churn'].map({'Yes': 1, 'No': 0})

# 2. One-Hot Encode the remaining text features
data = pd.get_dummies(data, drop_first=True)

# 3. Force all new 'bool' columns to pure integers (1s and 0s)
bool_cols = data.select_dtypes(include='bool').columns
data[bool_cols] = data[bool_cols].astype(int)

# 4. Separate Target (y) and Features (X)
X = data.drop('Churn', axis=1)
y = data['Churn']

# 5. The 80/20 Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 6. Scale the Data (Fit ONLY on training data, transform both)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Features encoded, split, and scaled perfectly.")

Features encoded, split, and scaled perfectly.


In [116]:
# Initialize SMOTE
smote = SMOTE(random_state=42)

# Generate synthetic churning customers STRICTLY on the training data
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)

print("SMOTE applied. Training data is now perfectly 50/50 balanced.")

SMOTE applied. Training data is now perfectly 50/50 balanced.


In [118]:
# 1. Initialize the optimized, GPU-accelerated XGBoost model
best_xgb = xgb.XGBClassifier(
    max_depth=3,
    learning_rate=0.01,
    n_estimators=100,
    eval_metric='logloss',
    tree_method='hist',
    device='cuda', # Forces the math onto the dedicated GPU
    random_state=42
)

# 2. Train the model on the balanced SMOTE data
best_xgb.fit(X_train_smote, y_train_smote)

# 3. Extract the raw probabilities from the Final Exam (Test Data)
probabilities = best_xgb.predict_proba(X_test_scaled)[:, 1]

# 4. Apply the Custom Business Threshold (35%)
custom_threshold = 0.35
custom_predictions = (probabilities >= custom_threshold).astype(int)

# 5. Print the Final Executive Report Card
print(f"--- Final Enterprise Model (Threshold: {custom_threshold}) ---")
print(classification_report(y_test, custom_predictions))

--- Final Enterprise Model (Threshold: 0.35) ---
              precision    recall  f1-score   support

           0       0.97      0.44      0.61      1036
           1       0.38      0.96      0.55       373

    accuracy                           0.58      1409
   macro avg       0.68      0.70      0.58      1409
weighted avg       0.82      0.58      0.59      1409

